# 10. Historial laboral de personas

Proyecto de tesis (Maestria en Ciencia de Datos e IA, ESPOL): *Sistema de Generacion de Perfiles del Personal Docente y Administrativo en ESPOL para la asignacion inteligente de tareas*. Este notebook corresponde a la **Fase 1 y 2** de la metodologia: extraccion y depuracion de una de las fuentes institucionales que alimentan el catalogo de variables (historia laboral, capacitaciones, proyectos, experiencia y direcciones de tesis) usado para construir los perfiles multidimensionales del personal.

**Fuente:** `data/raw/historialaboralpersonas.csv`  
**Salida:** `data/processed/historial_laboral_personas.csv`, `data/processed/historial_laboral_periodos_continuos.csv`, `data/processed/historial_laboral_features.csv`

Historial de contratos del personal en ESPOL (cargo, tipo de contrato, fechas, unidad). En este extracto contiene los contratos de una persona a modo de muestra.

`IDPERSONA`, `IDCONTRATOLABORAL`, `NOMBRETABLA`, `IDUBICACIONFISICAEMP`, `TIPOEMPLEADO`, `NOMBRE_UNIDAD` (renombrada desde `NOMBRE` del CSV crudo: nombre de la unidad) e `IDESTRUCTURAORGANICA` se conservan siempre (no se eliminan aunque resulten constantes en la muestra) porque son claves o categorias necesarias para cruzar este historial con el resto de fuentes del proyecto. `NOMBRE_UNIDAD` se rellena con `DESCONOCIDA` cuando llega vacío (ver sección 3). `TIPOEMPLEADO` e `IDREGIMENLABORAL` se decodifican en columnas `_DESC` adicionales (ver seccion 6). La sección 10 genera un subdataset con los periodos continuos de vinculación por persona, y la sección 11 una tabla de features por persona (antigüedad, evolución de RMU, cargos, dedicación docente, movilidad, régimen) lista para clustering/perfilamiento.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import _preprocesamiento_comun as pc

pd.set_option('display.max_columns', 100)

## 1. Carga de datos crudos

In [ ]:
df = pc.leer_csv('historialaboralpersonas.csv', low_memory=False)
df.head()

## 2. Exploración inicial

In [ ]:
pc.resumen(df, 'historial_laboral_personas')

In [ ]:
df.dtypes

### Valores atípicos en RMU

Antes de limpiar, se revisan los casos más extremos de `RMU` con el método del rango intercuartílico (`pc.detectar_outliers_iqr`, factor 1.5, el criterio clásico de boxplot) para decidir si son errores de digitación, casos legítimos poco frecuentes (p.ej. autoridades, cargos directivos) u otra cosa. El mismo método sirve para revisar otras columnas numéricas si hiciera falta (`HORAS`, etc.).

In [ ]:
print(df['RMU'].describe())

mask_outliers, limite_inferior, limite_superior = pc.detectar_outliers_iqr(df, 'RMU')
print(f"\nLimites IQR (factor 1.5): [{limite_inferior:.2f}, {limite_superior:.2f}]")
print(f"Casos atipicos: {mask_outliers.sum()} de {len(df)}")

columnas_revision = [c for c in ['IDPERSONA', 'IDCONTRATOLABORAL', 'CARGO', 'FECHAINICIOCONTRATO', 'RMU'] if c in df.columns]
df.loc[mask_outliers, columnas_revision].sort_values('RMU')

Estos casos son solo para revisión manual — no se eliminan ni se corrigen automáticamente, porque un RMU alto puede ser perfectamente legítimo (autoridades, cargos directivos, docentes titulares con muchos años) y no necesariamente un error. Si al revisar `IDCONTRATOLABORAL` contra la fuente institucional se confirma un error real (p.ej. un digito de más), corríjase puntualmente aquí antes de continuar con la limpieza, por ejemplo:

```python
correcciones_rmu = {}  # {IDCONTRATOLABORAL: valor_correcto}
if correcciones_rmu:
    df['RMU'] = df['IDCONTRATOLABORAL'].map(correcciones_rmu).fillna(df['RMU'])
```

## 3. Limpieza

`NOMBRE` (nombre de la unidad) se renombra a `NOMBRE_UNIDAD` para mayor claridad, y se rellena con `DESCONOCIDA` cuando llega vacío en el CSV crudo, en vez de eliminarse o dejarse nulo, para no perder la fila en agregaciones por unidad más adelante. `IDESTRUCTURAORGANICA` (identificador vigente de unidad, más confiable que `IDUNIDAD`/`IDUNIDADACADEMICAFIS`) también se protege de eliminarse por constante.

In [ ]:
df = df.rename(columns={'NOMBRE': 'NOMBRE_UNIDAD'})
df = pc.limpiar_strings(df)
df = pc.rellenar_categoricas_nulas(df, ['NOMBRE_UNIDAD'], valor='DESCONOCIDA')
df = pc.quitar_columnas_vacias(df, umbral=0.99)
df = pc.quitar_columnas_constantes(
    df,
    excluir=[
        'IDPERSONA', 'IDCONTRATOLABORAL', 'NOMBRETABLA', 'IDUBICACIONFISICAEMP',
        'TIPOEMPLEADO', 'NOMBRE_UNIDAD', 'IDESTRUCTURAORGANICA',
    ],
)

## 5. Tipado de fechas e identificadores

In [ ]:
df = pc.castear_fechas(df, ['FECHAINICIOCONTRATO', 'FECHAFINCONTRATO', 'FECHAVIGENCIA', 'FECHADESVINCULACION', 'FECHAINGRESOESPOL', 'FECHAREGISTRO'])
df = pc.castear_enteros(df, ['IDPERSONA', 'IDCONTRATOLABORAL', 'IDCARGO', 'IDUNIDAD', 'IDUBICACIONFISICAEMP', 'IDUSUARIO', 'IDUNIDADACADEMICAFIS', 'IDESTRUCTURAORGANICA'])

Nota: `FECHAFINCONTRATO` (y otras fechas de fin/desvinculacion) quedan como `NaT` cuando no existen en la fuente; eso indica que el contrato sigue vigente y no se elimina ni se invalida la fila.

## 6. Decodificación de catálogos

`TIPOEMPLEADO` (AA=Administrativo, DD=Docente) e `IDREGIMENLABORAL` (0=Contrato civil, 7=LOSEP, 8=LOES, 9=Código de Trabajo) se decodifican en columnas `_DESC` adicionales.

In [ ]:
df = pc.decodificar_historial_laboral(df)
df[['TIPOEMPLEADO', 'TIPOEMPLEADO_DESC', 'IDREGIMENLABORAL', 'IDREGIMENLABORAL_DESC']].drop_duplicates()

## 7. Verificación final

In [ ]:
pc.resumen(df, 'historial_laboral_personas (procesado)')
df.head()

In [ ]:
# Snapshot con todas las columnas (antes de la seccion 8) para el feature
# engineering de la seccion 11, que necesita HORAS/TIPODEDICACION/IDUNIDAD/etc.
df_historial = df.copy()

## 8. Columnas no usadas

Se eliminan columnas que no aportan al perfilamiento (identificadores internos de sistema, campos operativos/administrativos de bajo valor analitico o redundantes con otras ya conservadas):

`IDCARGO`, `IDUNIDAD`, `MODULO`, `HORARIOFLEXIBLE`, `FECHAVIGENCIA`, `HORAS`, `FECHAINGRESOESPOL`, `ESPORCONCURSO`, `REFARCHIVO1`, `REFARCHIVO2`, `FECHAREGISTRO`, `IDUSUARIO`, `TIPOVINCULO`, `IDUNIDADACADEMICAFIS`

In [ ]:
columnas_no_usadas = [
    'IDCARGO', 'IDUNIDAD', 'MODULO', 'HORARIOFLEXIBLE', 'FECHAVIGENCIA',
    'HORAS', 'FECHAINGRESOESPOL', 'ESPORCONCURSO', 'REFARCHIVO1', 'REFARCHIVO2',
    'FECHAREGISTRO', 'IDUSUARIO', 'TIPOVINCULO', 'IDUNIDADACADEMICAFIS',
]
df = df.drop(columns=[c for c in columnas_no_usadas if c in df.columns])
df.head()

In [ ]:
pc.resumen(df, 'historial_laboral_personas (procesado)')
df.head()

## 9. Guardado en data/processed

In [ ]:
pc.guardar_procesado(df, 'historial_laboral_personas.csv')

## 10. Periodos continuos de contratación

Fusiona los contratos de cada `IDPERSONA` en periodos continuos de vinculación con ESPOL, replicando la lógica de "historia laboral continua" del sistema origen (`AuxiliarObtenerHistoriaContinua`), implementada en `pc.calcular_periodos_continuos`:

- La fecha fin efectiva de cada contrato es `FECHADESVINCULACION` si existe; si no, `FECHAFINCONTRATO`; si ninguna existe, el contrato está vigente (sin fecha fin).
- Los contratos de cada persona se procesan ordenados por `FECHAINICIOCONTRATO`. Dos contratos consecutivos se fusionan en el mismo periodo si se solapan (un contrato contenido dentro del periodo actual se ignora, sin retroceder el fin), son exactamente contiguos (fin + 1 día = inicio siguiente), o dejan una brecha corta tolerada: el contrato termina en los últimos 3 días de un mes y el siguiente inicia el día 1 del mes calendario siguiente (corte administrativo de fin de mes, igual que en el sistema origen).
- Un periodo vigente (sin fecha fin) se preserva como vigente al fusionar; una fecha fin real nunca reemplaza una vigencia ya detectada.
- Cualquier otra brecha cierra el periodo actual y abre uno nuevo.

Salida: `data/processed/historial_laboral_periodos_continuos.csv`, un registro por periodo continuo (no por contrato).

In [ ]:
periodos_continuos = pc.calcular_periodos_continuos(df)
pc.guardar_procesado(periodos_continuos, 'historial_laboral_periodos_continuos.csv')
periodos_continuos.sort_values(['IDPERSONA', 'PERIODO_INICIO']).head(10)

In [ ]:
# Diagnostico: cuantos periodos continuos resulto teniendo cada persona
# (1 = historial sin brechas reales; >1 = hubo al menos una desvinculacion real entre contratos)
periodos_continuos.groupby('IDPERSONA').size().value_counts().sort_index().rename('n_personas').rename_axis('n_periodos_continuos')

## 11. Features de historial laboral para clustering/perfilamiento

A partir de `df_historial` (contratos, con todas las columnas de origen) y `periodos_continuos` (sección 10), se construye una tabla con **una fila por `IDPERSONA`** — la granularidad que necesita el modelo de perfilamiento — usando `pc.construir_features_historial_laboral`:

- **Conteo de contratos:** `N_REGISTROS_HISTORIAL` es la cantidad cruda de filas del CSV para la persona (puede incluir movimientos administrativos que no son un contrato nuevo). `N_CONTRATOS_TOTAL` es más representativo: cuenta cambios reales de `CARGO` o de `RMU` a lo largo de la línea de tiempo — dos filas consecutivas con el mismo cargo y la misma RMU cuentan como un solo contrato, en vez de contar filas. (Esto solo usa RMU como señal interna de "hubo un cambio", no expone el monto — ver nota de RMU más abajo.)
- **Antigüedad y continuidad:** fecha de primer ingreso, antigüedad efectiva (suma de días de los `PERIODO_DURACION_DIAS` de `periodos_continuos`, sin contar brechas reales) y antigüedad de calendario, número de periodos continuos y de reingresos, y si tiene vinculación vigente — **todo calculado a partir de `periodos_continuos`, no de las fechas crudas de cada contrato**, para no confundir vacaciones/licencias u otras brechas cortas ya fusionadas (sección 10) con una desvinculación real.
- **Tipo de empleado:** rol actual, si ha sido docente y administrativo a la vez, y años de experiencia aproximados por rol (`ANIOS_EXPERIENCIA_DOCENTE` / `ANIOS_EXPERIENCIA_ADMINISTRATIVO`).

  *Cómo leer estas columnas:* es la suma de días de todos los contratos de ese rol (fin efectivo — o hoy si sigue vigente — menos inicio), dividida entre 365.25 (promedia los años bisiestos) y redondeada a 2 decimales. Es un número decimal de años, **no** años+meses: p.ej. `5.01` son ~5 años y ~4 días (`0.01 × 365.25 ≈ 3.65` días), y `0.50` equivale a medio año (~6 meses). Puede sobreestimarse levemente si hay contratos simultáneos del mismo rol, ya que no se fusiona solapamiento dentro de un mismo rol (a diferencia de la antigüedad general, que sí usa `periodos_continuos`).
- **Dedicación docente:** la más reciente, la más frecuente y cuántas distintas tuvo (solo si `TIPODEDICACION` sobrevive la limpieza, p.ej. no en muestras solo administrativas).
- **Cargos:** cantidad de cargos distintos, cargo actual, cargo más frecuente, y si en algún año calendario tuvo más de un cargo distinto (`MULTIPLES_CARGOS_MISMO_ANIO`).
- **Movilidad:** cantidad de unidades distintas por `IDESTRUCTURAORGANICA` (no por `IDUNIDAD`/`IDUNIDADACADEMICAFIS`, identificadores de estructuras anteriores menos confiables); nombre de la unidad actual (`NOMBRE_UNIDAD`); cantidad de facultades distintas y si en algún momento pasó por rectorado/vicerrectorado, detectado por texto en `NOMBRE_UNIDAD` (`"FACULTAD"` / `"RECTORADO"`).
- **Régimen laboral:** régimen inicial y actual, y cantidad de regímenes distintos por los que pasó.
- **RMU: omitida por ahora (`incluir_rmu=False`).** El histórico mezcla montos en sucres y en dólares por la transición monetaria de Ecuador (año 2000), sin una conversión/normalización todavía implementada — comparar o promediar RMU sin resolver eso primero produciría magnitudes y "crecimientos" sin sentido. `pc.construir_features_historial_laboral` soporta `incluir_rmu=True` para cuando se resuelva la normalización de moneda; mientras tanto, las columnas `RMU_*` no se incluyen en el dataset exportado.
- **Estabilidad contractual:** `PROPORCION_CONTRATOS_FINALIZADOS` = proporción de `ESTADOCONTRATO`='FF' **solo entre las filas con `TIPO`='V'** (vinculación), si la columna `TIPO` existe. `TIPO`='M' suele ser un movimiento (vacaciones, licencias, etc.) y no una desvinculación real, pero esa clasificación aún no está depurada del todo — por ahora solo se filtra por 'V', sin más tratamiento.

Limitaciones conocidas:
- Los años de experiencia por rol pueden sobreestimarse levemente si existen contratos simultáneos del mismo `TIPOEMPLEADO` (ver arriba).
- La detección de facultad/rectorado por texto en `NOMBRE_UNIDAD` es una heurística simple (substring, sin distinguir mayúsculas); conviene revisarla contra los valores reales una vez que se disponga de datos de más personas.
- La clasificación `TIPO`='M' (movimientos) aún no distingue vacaciones, licencias u otros casos; queda pendiente para una futura depuración.
- RMU queda pendiente de una normalización sucre/dólar antes de reincorporarse como feature.

Salida: `data/processed/historial_laboral_features.csv`, un registro por persona.

In [ ]:
features_historial = pc.construir_features_historial_laboral(df_historial, periodos_continuos, incluir_rmu=False)
pc.guardar_procesado(features_historial, 'historial_laboral_features.csv')
features_historial.head()

In [ ]:
pc.resumen(features_historial, 'historial_laboral_features')